[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/080.%20The%20International%20Freight%20Mode%20Selection%20Problem%20%28Air%20vs.%20Ocean%29/P80-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 80. The International Freight Mode Selection Problem (Air vs. Ocean)
## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Key assumptions
- Population-based search can discover non-intuitive optimal patterns
- Genetic operators preserve feasibility while exploring diverse solutions
- Multi-objective fitness balances cost, capacity, and service quality
- Convergence analysis guides parameter tuning and stopping criteria
- Metaheuristic can handle complex interdependencies between shipments

### Approach (step-by-step)
1. **Initialize population**: Mix of heuristic-based and random solutions
2. **Evaluate fitness**: Calculate total cost with capacity penalty
3. **Select parents**: Tournament selection for pressure preservation
4. **Apply crossover**: Two-point crossover with capacity repair
5. **Apply mutation**: Bit-flip mutation with feasibility restoration
6. **Evolve generations**: Elitism preserves best solutions
7. **Analyze convergence**: Track fitness improvement and diversity

### What to look for in the results
- Population evolution and fitness improvement over generations
- Convergence behavior and solution quality metrics
- Discovery of non-obvious shipment patterns
- Performance comparison with heuristic and optimal methods

### Concrete example (from the source)
The Genetic Algorithm is tested on a 50-shipment optimization problem:
- Population size: 100 individuals
- Generations: 200 evolution cycles
- Crossover rate: 80%, Mutation rate: 10%
- Elite ratio: 20% best individuals preserved

Expected results: GA discovers solution with total cost $892,450 vs $931,200 from greedy heuristic (4.2% improvement). The algorithm identifies patterns like using air freight for medium-value shipments to enable better ocean freight utilization for heavier cargo, demonstrating population-based search advantages.

In [ ]:
# Import required packages for genetic algorithm implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Shipment:
    """Data class for shipment information"""
    id: int
    weight: float  # kg
    value: float   # USD
    destination: str
    due_date: int  # days from now

@dataclass
class TransportParams:
    """Parameters for transportation costs and times"""
    costs: Dict[str, Dict[str, float]]  # mode -> destination -> cost per kg
    transit_times: Dict[str, Dict[str, int]]  # mode -> destination -> days
    holding_rate: float  # daily holding cost rate as fraction of value
    late_penalty: float  # penalty per day late
    air_capacity: float  # monthly air freight capacity in kg

@dataclass
class GAParameters:
    """Genetic Algorithm parameters"""
    population_size: int = 100
    generations: int = 200
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    elite_ratio: float = 0.2
    tournament_size: int = 3

# Create a larger test instance for GA demonstration
def create_test_instance(num_shipments: int = 50) -> Tuple[List[Shipment], TransportParams]:
    """
    Create a test instance with multiple shipments for GA optimization
    """
    random.seed(42)  # For reproducibility

    # Generate diverse shipments
    shipments = []
    destinations = ['rotterdam', 'los_angeles', 'singapore', 'hamburg', 'new_york']

    for i in range(num_shipments):
        # Create varied shipment characteristics
        weight = random.uniform(100, 8000)  # 100kg to 8000kg
        value = random.uniform(50000, 3000000)  # $50K to $3M
        destination = random.choice(destinations)
        due_date = random.randint(5, 45)  # 5 to 45 days

        shipments.append(Shipment(i + 1, weight, value, destination, due_date))

    # Define transportation parameters
    params = TransportParams(
        costs={
            'ocean': {
                'rotterdam': 2.2, 'los_angeles': 1.9, 'singapore': 2.5,
                'hamburg': 2.1, 'new_york': 1.7
            },
            'air': {
                'rotterdam': 6.5, 'los_angeles': 5.8, 'singapore': 7.2,
                'hamburg': 6.3, 'new_york': 5.5
            }
        },
        transit_times={
            'ocean': {
                'rotterdam': 28, 'los_angeles': 20, 'singapore': 32,
                'hamburg': 26, 'new_york': 18
            },
            'air': {
                'rotterdam': 3, 'los_angeles': 2, 'singapore': 4,
                'hamburg': 3, 'new_york': 2
            }
        },
        holding_rate=0.0012,  # 0.12% per day
        late_penalty=1200,   # $1200 per day
        air_capacity=25000  # 25000 kg per month
    )

    return shipments, params

# Create test instance
shipments, params = create_test_instance(50)
ga_params = GAParameters()

print(f"Test Instance Created:")
print(f"  Number of shipments: {len(shipments)}")
print(f"  Total weight: {sum(s.weight for s in shipments):,.0f} kg")
print(f"  Total value: ${sum(s.value for s in shipments):,.0f}")
print(f"  Air capacity: {params.air_capacity:,} kg")
print(f"  GA parameters: pop={ga_params.population_size}, gen={ga_params.generations}")

# Show sample shipments
print(f"\nSample shipments:")
for i, s in enumerate(shipments[:5]):
    print(f"  {s.id}: {s.weight:.0f}kg, ${s.value:,.0f}, to {s.destination}, due {s.due_date} days")

Test Instance Created:
  Number of shipments: 50
  Total weight: 193,557 kg
  Total value: $81,245,872
  Air capacity: 25,000 kg
  GA parameters: pop=100, gen=200

Sample shipments:
  1: 5151kg, $123,782, to singapore, due 20 days
  2: 1863kg, $2,222,590, to new_york, due 10 days
  3: 4765kg, $143,759, to rotterdam, due 18 days
  4: 1938kg, $1,825,955, to new_york, due 17 days
  5: 5757kg, $2,118,909, to hamburg, due 19 days


In [ ]:
try:
    class FreightModeGA:
        """
        Genetic Algorithm for freight mode selection optimization
        Implements the GA design from the source material
        """
    
        def __init__(self, shipments: List[Shipment], params: TransportParams, ga_params: GAParameters):
            self.shipments = shipments
            self.params = params
            self.ga_params = ga_params
            self.n = len(shipments)
    
            # Initialize priority selector for heuristic individuals
            from P80_Tier_2 import FreightModeSelector
            self.priority_selector = FreightModeSelector(params)
    
            # Track evolution statistics
            self.evolution_history = {
                'best_fitness': [],
                'avg_fitness': [],
                'diversity': [],
                'best_solutions': []
            }
    
        def calculate_total_cost(self, shipment: Shipment, mode: str) -> float:
            """
            Calculate total cost for a single shipment using a specific transport mode
            """
            # Transportation cost
            transport_cost = shipment.weight * self.params.costs[mode][shipment.destination]
    
            # Transit time
            transit_time = self.params.transit_times[mode][shipment.destination]
    
            # Holding cost if arrives early
            if transit_time < shipment.due_date:
                early_days = shipment.due_date - transit_time
                holding_cost = early_days * self.params.holding_rate * shipment.value
            else:
                holding_cost = 0
    
            # Late penalty if arrives after due date
            if transit_time > shipment.due_date:
                late_days = transit_time - shipment.due_date
                late_penalty_cost = late_days * self.params.late_penalty
            else:
                late_penalty_cost = 0
    
            return transport_cost + holding_cost + late_penalty_cost
    
        def initialize_population(self) -> List[List[int]]:
            """
            Create initial population: 25% heuristic-based, 75% random plans
            """
            population = []
    
            # Heuristic-based individuals (25% of population)
            heuristic_count = int(self.ga_params.population_size * 0.25)
            for _ in range(heuristic_count):
                individual = self._create_heuristic_individual()
                population.append(individual)
    
            # Random individuals (75% of population)
            for _ in range(self.ga_params.population_size - heuristic_count):
                individual = [random.randint(0, 1) for _ in range(self.n)]  # 0: ocean, 1: air
                population.append(individual)
    
            return population
    
        def _create_heuristic_individual(self) -> List[int]:
            """
            Create individual based on priority scores (prefer air for urgent shipments)
            """
            individual = []
            for shipment in self.shipments:
                priority, preferred_mode = self.priority_selector.calculate_priority_score(shipment)
                individual.append(1 if preferred_mode == 'air' else 0)
            return individual
    
        def evaluate_fitness(self, individual: List[int]) -> float:
            """
            Calculate negative total cost (for maximization); penalize air capacity violation
            """
            total_cost = 0
            air_weight = 0
    
            for i, mode_gene in enumerate(individual):
                shipment = self.shipments[i]
                mode_str = 'air' if mode_gene == 1 else 'ocean'
    
                if mode_gene == 1:
                    air_weight += shipment.weight
    
                total_cost += self.calculate_total_cost(shipment, mode_str)
    
            # Add penalty for air capacity violation
            if air_weight > self.params.air_capacity:
                penalty = (air_weight - self.params.air_capacity) * 10  # $10 per kg over capacity
                total_cost += penalty
    
            return -total_cost  # Negative for maximization
    
        def tournament_selection(self, population: List[List[int]], fitness: List[float]) -> List[int]:
            """
            Select best individual from random tournament
            """
            indices = random.sample(range(len(population)), self.ga_params.tournament_size)
            best_idx = indices[np.argmax([fitness[i] for i in indices])]
            return population[best_idx].copy()
    
        def order_crossover(self, parent1: List[int], parent2: List[int]) -> Tuple[List[int], List[int]]:
            """
            Two-point crossover with capacity repair
            """
            if random.random() > self.ga_params.crossover_rate:
                return parent1.copy(), parent2.copy()
    
            # Select two crossover points
            start, end = sorted(random.sample(range(1, self.n), 2))
    
            # Create offspring
            child1 = parent1.copy()
            child2 = parent2.copy()
    
            # Swap segments between crossover points
            child1[start:end] = parent2[start:end]
            child2[start:end] = parent1[start:end]
    
            # Repair capacity constraints
            child1 = self._repair_capacity(child1)
            child2 = self._repair_capacity(child2)
    
            return child1, child2
    
        def _repair_capacity(self, individual: List[int]) -> List[int]:
            """
            Repair air capacity by switching heaviest air shipments to ocean
            """
            air_weight = sum(self.shipments[i].weight for i, gene in enumerate(individual) if gene == 1)
    
            if air_weight <= self.params.air_capacity:
                return individual
    
            # Identify air shipments sorted by weight (heaviest first)
            air_shipments = [(i, self.shipments[i].weight)
                            for i, gene in enumerate(individual) if gene == 1]
            air_shipments.sort(key=lambda x: x[1], reverse=True)
    
            # Switch heaviest air shipments to ocean until capacity is satisfied
            repaired = individual.copy()
            for idx, weight in air_shipments:
                if air_weight <= self.params.air_capacity:
                    break
                repaired[idx] = 0  # Switch to ocean
                air_weight -= weight
    
            return repaired
    
        def mutate(self, individual: List[int]) -> List[int]:
            """
            Bit-flip mutation followed by capacity repair
            """
            mutated = []
            for gene in individual:
                if random.random() < self.ga_params.mutation_rate:
                    mutated.append(1 - gene)  # Flip bit
                else:
                    mutated.append(gene)
    
            return self._repair_capacity(mutated)
    
        def calculate_diversity(self, population: List[List[int]]) -> float:
            """
            Calculate population diversity as average Hamming distance
            """
            if len(population) < 2:
                return 0.0
    
            total_distance = 0
            comparisons = 0
    
            for i in range(len(population)):
                for j in range(i + 1, len(population)):
                    # Calculate Hamming distance
                    distance = sum(g1 != g2 for g1, g2 in zip(population[i], population[j]))
                    total_distance += distance
                    comparisons += 1
    
            return total_distance / comparisons if comparisons > 0 else 0.0
    
        def evolve(self) -> Tuple[List[int], float, Dict]:
            """
            Main GA evolution loop
            Returns best solution, fitness, and evolution statistics
            """
            start_time = time.time()
    
            # Initialize population
            population = self.initialize_population()
            best_individual = None
            best_fitness = float('-inf')
    
            print("Starting GA evolution...")
    
            for generation in range(self.ga_params.generations):
                # Evaluate fitness
                fitness = [self.evaluate_fitness(ind) for ind in population]
    
                # Update best solution
                current_best_fitness = max(fitness)
                if current_best_fitness > best_fitness:
                    best_fitness = current_best_fitness
                    best_individual = population[fitness.index(best_fitness)].copy()
    
                # Track evolution statistics
                avg_fitness = np.mean(fitness)
                diversity = self.calculate_diversity(population)
    
                self.evolution_history['best_fitness'].append(best_fitness)
                self.evolution_history['avg_fitness'].append(avg_fitness)
                self.evolution_history['diversity'].append(diversity)
    
                # Progress reporting
                if generation % 20 == 0 or generation == self.ga_params.generations - 1:
                    print(f"  Gen {generation:3d}: Best=${-best_fitness:,.0f}, Avg=${-avg_fitness:,.0f}, Diversity={diversity:.1f}")
    
                # Elitism: preserve top individuals
                elite_count = int(self.ga_params.population_size * self.ga_params.elite_ratio)
                elite_indices = sorted(range(len(fitness)), key=lambda x: -fitness[x])[:elite_count]
                elite = [population[i].copy() for i in elite_indices]
    
                # Generate new population
                new_population = elite.copy()
    
                while len(new_population) < self.ga_params.population_size:
                    # Selection
                    parent1 = self.tournament_selection(population, fitness)
                    parent2 = self.tournament_selection(population, fitness)
    
                    # Crossover
                    child1, child2 = self.order_crossover(parent1, parent2)
    
                    # Mutation
                    child1 = self.mutate(child1)
                    child2 = self.mutate(child2)
    
                    new_population.extend([child1, child2])
    
                # Trim to exact population size
                population = new_population[:self.ga_params.population_size]
    
            evolution_time = time.time() - start_time
    
            print(f"GA evolution completed in {evolution_time:.2f} seconds")
            print(f"Best solution cost: ${-best_fitness:,.0f}")
    
            return best_individual, -best_fitness, self.evolution_history
    
    # Initialize GA
    ga = FreightModeGA(shipments, params, ga_params)
    print("Genetic Algorithm initialized")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: No module named 'P80_Tier_2'...]

In [ ]:
try:
    # Run the genetic algorithm
    best_solution, best_cost, evolution_stats = ga.evolve()
    
    print("\nGA Optimization Results:")
    print("=" * 80)
    print(f"Best solution cost: ${best_cost:,.0f}")
    print(f"Number of air shipments: {sum(best_solution)}")
    print(f"Number of ocean shipments: {len(best_solution) - sum(best_solution)}")
    
    # Calculate air weight utilization
    air_weight = sum(shipments[i].weight for i, gene in enumerate(best_solution) if gene == 1)
    print(f"Air weight used: {air_weight:,} kg / {params.air_capacity:,} kg")
    print(f"Capacity utilization: {air_weight/params.air_capacity*100:.1f}%")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ga' is not defined...]

In [ ]:
try:
    # Compare with baseline strategies
    def calculate_baseline_costs(shipments: List[Shipment], params: TransportParams) -> Dict[str, float]:
        """
        Calculate costs for baseline strategies
        """
        costs = {}
    
        # All ocean
        ocean_cost = sum(ga.calculate_total_cost(s, 'ocean') for s in shipments)
        costs['all_ocean'] = ocean_cost
    
        # All air (check capacity)
        total_air_weight = sum(s.weight for s in shipments)
        if total_air_weight <= params.air_capacity:
            air_cost = sum(ga.calculate_total_cost(s, 'air') for s in shipments)
            costs['all_air'] = air_cost
        else:
            costs['all_air'] = float('inf')
    
        # Greedy heuristic
        heuristic_assignments, _ = ga.priority_selector.select_modes_batch(shipments)
        heuristic_cost = sum(ga.calculate_total_cost(s, heuristic_assignments[s.id]) for s in shipments)
        costs['heuristic'] = heuristic_cost
    
        return costs
    
    # Calculate baseline costs
    baseline_costs = calculate_baseline_costs(shipments, params)
    
    print("\nStrategy Comparison:")
    print("=" * 80)
    strategies = {
        'All Ocean': baseline_costs['all_ocean'],
        'Heuristic': baseline_costs['heuristic'],
        'GA Solution': best_cost
    }
    
    if baseline_costs['all_air'] != float('inf'):
        strategies['All Air'] = baseline_costs['all_air']
    
    # Sort by cost
    sorted_strategies = sorted(strategies.items(), key=lambda x: x[1])
    
    for name, cost in sorted_strategies:
        if cost == float('inf'):
            print(f"{name}: Infeasible (exceeds capacity)")
        else:
            print(f"{name}: ${cost:,.0f}")
    
            # Calculate improvement vs heuristic
            if name != 'Heuristic':
                improvement = (baseline_costs['heuristic'] - cost) / baseline_costs['heuristic'] * 100
                print(f"  Improvement vs Heuristic: {improvement:+.1f}%")
    
    # Analyze GA solution characteristics
    print(f"\nGA Solution Analysis:")
    print(f"  Cost improvement vs heuristic: {(baseline_costs['heuristic'] - best_cost) / baseline_costs['heuristic'] * 100:+.1f}%")
    print(f"  Cost improvement vs all-ocean: {(baseline_costs['all_ocean'] - best_cost) / baseline_costs['all_ocean'] * 100:+.1f}%")
    
    # Analyze shipment patterns in GA solution
    air_shipments = [shipments[i] for i, gene in enumerate(best_solution) if gene == 1]
    ocean_shipments = [shipments[i] for i, gene in enumerate(best_solution) if gene == 0]
    
    if air_shipments:
        avg_air_value_density = np.mean([s.value / s.weight for s in air_shipments])
        avg_air_urgency = np.mean([max(0, 10 - (s.due_date - params.transit_times['air'][s.destination]) / 3) for s in air_shipments])
        print(f"\nAir shipment characteristics:")
        print(f"  Average value density: ${avg_air_value_density:,.0f}/kg")
        print(f"  Average urgency score: {avg_air_urgency:.2f}")
        print(f"  Weight range: {min(s.weight for s in air_shipments):,.0f} - {max(s.weight for s in air_shipments):,.0f} kg")
    
    if ocean_shipments:
        avg_ocean_value_density = np.mean([s.value / s.weight for s in ocean_shipments])
        print(f"\nOcean shipment characteristics:")
        print(f"  Average value density: ${avg_ocean_value_density:,.0f}/kg")
        print(f"  Weight range: {min(s.weight for s in ocean_shipments):,.0f} - {max(s.weight for s in ocean_shipments):,.0f} kg")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ga' is not defined...]

In [ ]:
try:
    # Create comprehensive GA visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Genetic Algorithm Optimization Analysis', fontsize=16, fontweight='bold')
    
    # 1. Convergence analysis
    generations = range(len(evolution_stats['best_fitness']))
    best_costs = [-f for f in evolution_stats['best_fitness']]
    avg_costs = [-f for f in evolution_stats['avg_fitness']]
    
    ax1.plot(generations, best_costs, 'b-', linewidth=2, label='Best Fitness')
    ax1.plot(generations, avg_costs, 'r--', linewidth=1, alpha=0.7, label='Average Fitness')
    ax1.set_xlabel('Generation')
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('GA Convergence Analysis')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Add heuristic baseline
    ax1.axhline(y=baseline_costs['heuristic'], color='orange', linestyle=':',
                label=f'Heuristic (${baseline_costs["heuristic"]:,.0f})', linewidth=2)
    ax1.axhline(y=baseline_costs['all_ocean'], color='green', linestyle=':',
                label=f'All Ocean (${baseline_costs["all_ocean"]:,.0f})', linewidth=1)
    ax1.legend()
    
    # 2. Population diversity over time
    ax2.plot(generations, evolution_stats['diversity'], 'g-', linewidth=2, markersize=4)
    ax2.set_xlabel('Generation')
    ax2.set_ylabel('Average Hamming Distance')
    ax2.set_title('Population Diversity Evolution')
    ax2.grid(True, alpha=0.3)
    
    # 3. Strategy cost comparison
    strategy_names = []
    strategy_costs = []
    strategy_colors = []
    
    for name, cost in sorted_strategies:
        if cost != float('inf'):
            strategy_names.append(name.replace(' ', '\n'))
            strategy_costs.append(cost)
            if 'GA' in name:
                strategy_colors.append('blue')
            elif 'Heuristic' in name:
                strategy_colors.append('orange')
            elif 'Ocean' in name:
                strategy_colors.append('green')
            else:
                strategy_colors.append('red')
    
    bars = ax3.bar(strategy_names, strategy_costs, color=strategy_colors, alpha=0.7)
    ax3.set_ylabel('Total Cost ($)')
    ax3.set_title('Strategy Cost Comparison')
    ax3.tick_params(axis='x', rotation=45)
    
    # Add value labels and improvement percentages
    heuristic_cost = baseline_costs['heuristic']
    for bar, cost, name in zip(bars, strategy_costs, strategy_names):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                 f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
    
        # Add improvement percentage
        if 'GA' in name and cost != heuristic_cost:
            improvement = (heuristic_cost - cost) / heuristic_cost * 100
            ax3.text(bar.get_x() + bar.get_width()/2., height - height*0.05,
                     f'{improvement:+.1f}%', ha='center', va='top', fontweight='bold', color='blue')
    
    ax3.grid(True, alpha=0.3)
    
    # 4. Shipment mode distribution
    mode_counts = {'Air': sum(best_solution), 'Ocean': len(best_solution) - sum(best_solution)}
    mode_weights = {
        'Air': sum(shipments[i].weight for i, gene in enumerate(best_solution) if gene == 1),
        'Ocean': sum(shipments[i].weight for i, gene in enumerate(best_solution) if gene == 0)
    }
    
    # Create stacked bar chart
    categories = ['Air Shipments', 'Ocean Shipments']
    counts = [mode_counts['Air'], mode_counts['Ocean']]
    weights = [mode_weights['Air'], mode_weights['Ocean']]
    
    # Normalize weights for visualization
    max_weight = max(weights)
    normalized_weights = [w/max_weight * max(counts) * 0.8 for w in weights]
    
    ax4.bar(categories, counts, color=['lightcoral', 'lightblue'], alpha=0.7, label='Shipment Count')
    ax4.bar(categories, normalized_weights, color=['red', 'blue'], alpha=0.3, label='Weight (scaled)')
    
    ax4.set_ylabel('Count / Scaled Weight')
    ax4.set_title('GA Solution: Mode Distribution')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Add count labels
    for i, (category, count, weight) in enumerate(zip(categories, counts, weights)):
        ax4.text(i, count + max(counts)*0.02, f'{count} shipments\n{weight:,.0f} kg',
                 ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nVisualization Summary:")
    print("=" * 80)
    print(f"1. Convergence: Shows GA improvement over generations vs baselines")
    print(f"2. Diversity: Population diversity evolution indicating exploration vs exploitation")
    print(f"3. Comparison: GA solution cost vs heuristic and baseline strategies")
    print(f"4. Distribution: Mode assignment patterns in GA optimal solution")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'evolution_stats' is not defined...]

In [ ]:
try:
    # Advanced analysis: Discover non-intuitive patterns
    def analyze_solution_patterns(shipments: List[Shipment], solution: List[int], params: TransportParams) -> Dict:
        """
        Analyze patterns in the GA solution to identify non-intuitive decisions
        """
        air_shipments = []
        ocean_shipments = []
    
        for i, gene in enumerate(solution):
            if gene == 1:
                air_shipments.append(shipments[i])
            else:
                ocean_shipments.append(shipments[i])
    
        analysis = {
            'air_stats': {},
            'ocean_stats': {},
            'non_intuitive_cases': []
            'capacity_efficiency': 0
        }
    
        # Air shipment statistics
        if air_shipments:
            air_value_densities = [s.value / s.weight for s in air_shipments]
            air_urgencies = []
            for s in air_shipments:
                time_buffer = s.due_date - params.transit_times['ocean'][s.destination]
                urgency = max(0, 10 - time_buffer / 3)
                air_urgencies.append(urgency)
    
            analysis['air_stats'] = {
                'count': len(air_shipments),
                'avg_value_density': np.mean(air_value_densities),
                'avg_urgency': np.mean(air_urgencies),
                'total_weight': sum(s.weight for s in air_shipments)
            }
    
        # Ocean shipment statistics
        if ocean_shipments:
            ocean_value_densities = [s.value / s.weight for s in ocean_shipments]
            ocean_urgencies = []
            for s in ocean_shipments:
                time_buffer = s.due_date - params.transit_times['ocean'][s.destination]
                urgency = max(0, 10 - time_buffer / 3)
                ocean_urgencies.append(urgency)
    
            analysis['ocean_stats'] = {
                'count': len(ocean_shipments),
                'avg_value_density': np.mean(ocean_value_densities),
                'avg_urgency': np.mean(ocean_urgencies),
                'total_weight': sum(s.weight for s in ocean_shipments)
            }
    
        # Identify non-intuitive cases
        for i, shipment in enumerate(shipments):
            assigned_mode = 'air' if solution[i] == 1 else 'ocean'
    
            # Calculate heuristic preference
            priority, heuristic_mode = ga.priority_selector.calculate_priority_score(shipment)
    
            # Check if GA decision differs from heuristic
            if assigned_mode != heuristic_mode:
                analysis['non_intuitive_cases'].append({
                    'shipment_id': shipment.id,
                    'weight': shipment.weight,
                    'value': shipment.value,
                    'destination': shipment.destination,
                    'ga_mode': assigned_mode,
                    'heuristic_mode': heuristic_mode,
                    'priority_score': priority
                })
    
        # Calculate capacity efficiency
        total_air_weight = analysis['air_stats'].get('total_weight', 0)
        analysis['capacity_efficiency'] = total_air_weight / params.air_capacity
    
        return analysis
    
    # Perform pattern analysis
    pattern_analysis = analyze_solution_patterns(shipments, best_solution, params)
    
    print("\nSolution Pattern Analysis:")
    print("=" * 80)
    
    print(f"Air Shipments:")
    if pattern_analysis['air_stats']:
        air_stats = pattern_analysis['air_stats']
        print(f"  Count: {air_stats['count']}")
        print(f"  Average value density: ${air_stats['avg_value_density']:,.0f}/kg")
        print(f"  Average urgency: {air_stats['avg_urgency']:.2f}")
        print(f"  Total weight: {air_stats['total_weight']:,.0f} kg")
    
    print(f"\nOcean Shipments:")
    if pattern_analysis['ocean_stats']:
        ocean_stats = pattern_analysis['ocean_stats']
        print(f"  Count: {ocean_stats['count']}")
        print(f"  Average value density: ${ocean_stats['avg_value_density']:,.0f}/kg")
        print(f"  Average urgency: {ocean_stats['avg_urgency']:.2f}")
        print(f"  Total weight: {ocean_stats['total_weight']:,.0f} kg")
    
    print(f"\nCapacity Utilization: {pattern_analysis['capacity_efficiency']*100:.1f}%")
    
    print(f"\nNon-Intuitive Decisions (GA vs Heuristic):")
    if pattern_analysis['non_intuitive_cases']:
        for case in pattern_analysis['non_intuitive_cases'][:10]:  # Show first 10
            print(f"  Shipment {case['shipment_id']}: GA={case['ga_mode']}, Heuristic={case['heuristic_mode']} (priority: {case['priority_score']:.1f})")
        print(f"  Total non-intuitive decisions: {len(pattern_analysis['non_intuitive_cases'])} ({len(pattern_analysis['non_intuitive_cases'])/len(shipments)*100:.1f}%)")
    else:
        print(f"  No non-intuitive decisions found")
    
    # Create pattern visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('GA Solution Pattern Analysis', fontsize=16, fontweight='bold')
    
    # Value density vs urgency scatter plot
    if pattern_analysis['air_stats'] and pattern_analysis['ocean_stats']:
        air_shipments = [shipments[i] for i, gene in enumerate(best_solution) if gene == 1]
        ocean_shipments = [shipments[i] for i, gene in enumerate(best_solution) if gene == 0]
    
        # Calculate metrics for scatter plot
        air_data = []
        for s in air_shipments:
            value_density = s.value / s.weight
            time_buffer = s.due_date - params.transit_times['ocean'][s.destination]
            urgency = max(0, 10 - time_buffer / 3)
            air_data.append((value_density, urgency))
    
        ocean_data = []
        for s in ocean_shipments:
            value_density = s.value / s.weight
            time_buffer = s.due_date - params.transit_times['ocean'][s.destination]
            urgency = max(0, 10 - time_buffer / 3)
            ocean_data.append((value_density, urgency))
    
        if air_data:
            air_x, air_y = zip(*air_data)
            ax1.scatter(air_x, air_y, c='red', alpha=0.7, s=50, label='Air')
    
        if ocean_data:
            ocean_x, ocean_y = zip(*ocean_data)
            ax1.scatter(ocean_x, ocean_y, c='blue', alpha=0.7, s=50, label='Ocean')
    
        ax1.set_xlabel('Value Density ($/kg)')
        ax1.set_ylabel('Urgency Score')
        ax1.set_title('Shipment Characteristics by Mode')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Non-intuitive decisions analysis
    if pattern_analysis['non_intuitive_cases']:
        cases = pattern_analysis['non_intuitive_cases'][:20]  # First 20 cases
        shipment_ids = [case['shipment_id'] for case in cases]
        priorities = [case['priority_score'] for case in cases]
    
        colors = ['red' if case['ga_mode'] == 'air' else 'blue' for case in cases]
    
        bars = ax2.bar(range(len(shipment_ids)), priorities, color=colors, alpha=0.7)
        ax2.set_xlabel('Shipment ID')
        ax2.set_ylabel('Priority Score')
        ax2.set_title('Non-Intuitive Decisions (GA vs Heuristic)')
        ax2.set_xticks(range(len(shipment_ids)))
        ax2.set_xticklabels([f'S{id}' for id in shipment_ids], rotation=45)
        ax2.axhline(y=50, color='green', linestyle='--', label='Priority Threshold', linewidth=2)
        ax2.grid(True, alpha=0.3)
        ax2.legend()
    else:
        ax2.text(0.5, 0.5, 'No Non-Intuitive\nDecisions Found',
                 ha='center', va='center', fontsize=14, transform=ax2.transAxes)
        ax2.set_title('Non-Intuitive Decisions Analysis')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: invalid syntax. Perhaps you forgot a comma? (<string>, line 18)...]

### Why this Tier exists vs earlier Tiers
The Genetic Algorithm addresses limitations of both mathematical and heuristic approaches:
- **Global optimization**: Escapes local optima that trap greedy heuristics
- **Pattern discovery**: Identifies non-intuitive shipment combinations
- **Scalable intelligence**: Maintains solution quality as problem size grows
- **Adaptive search**: Balances exploration and exploitation automatically
- **Complex interdependencies**: Handles subtle shipment interactions

### Pros / Cons vs Tiers 1-2
**Pros:**
- Discovers non-obvious optimal patterns (4.2% improvement vs heuristic)
- Handles complex interdependencies between shipments
- Scales well to large problems (100+ shipments)
- Provides convergence analysis and solution quality metrics
- Robust to local optima and solution space complexity

**Cons:**
- No optimality guarantees (stochastic algorithm)
- Computationally intensive vs heuristics (seconds vs milliseconds)
- Parameter tuning required for different problem types
- Solution quality varies between runs
- More complex implementation and debugging

### When to use this Tier
- **Large-scale optimization** where heuristics may get stuck in local optima
- **Complex interdependencies** between shipments that simple rules miss
- **Quality-critical applications** where small improvements have high value
- **Pattern discovery** for understanding shipment optimization dynamics
- **Benchmark development** for evaluating other optimization methods
- **Research and analysis** of freight mode selection behavior